# Agentic AI Fundamentals

## Notebook 09: Guardrails, Human Approval, and Failure Recovery

An agent connects probabilistic model decisions to deterministic application actions. Guardrails protect that boundary, but no single classifier or prompt can handle every risk. Reliable systems use several controls at different stages: input validation, tool allowlists, argument schemas, execution policy, human approval, bounded retries, and output inspection.

This notebook builds those controls around two tools. Reading a simulated account status is low-risk and may execute automatically. Sending an external notification changes state and must pause for explicit approval.

## Learning objectives

By the end of this notebook, you should be able to:

- place guardrails at distinct system boundaries;
- classify tools by risk and side effects;
- distinguish authorization from argument validation;
- pause and resume LangGraph execution for human approval;
- retry transient failures without retrying permanent failures;
- preserve idempotency considerations for side-effecting tools; and
- inspect output before returning it to a user.

## 1. Guardrails are a layered system

| Boundary | Example control |
|---|---|
| User input | Length limits and prohibited request checks |
| Model output | Structured schema validation |
| Tool selection | Explicit registry and per-user permissions |
| Tool arguments | Pydantic validation and domain rules |
| Side effects | Human approval and idempotency keys |
| Execution | Timeouts and bounded retry |
| Final output | Secret redaction and citation checks |

These controls address different failure modes. A valid schema does not grant authorization, and human approval does not make malformed arguments safe.

## 2. Install dependencies and configure Gemini

The policy and approval examples run locally. The final integration uses Gemini to propose the action and therefore requires `GEMINI_API_KEY` in Colab Secrets.

In [1]:
%pip install -q -U "langgraph>=1.0,<2" "google-genai>=1.0.0" "pydantic>=2.7,<3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [2]:
import os


def load_gemini_api_key() -> str:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except (ImportError, KeyError):
        api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets.")
    return api_key


GEMINI_API_KEY = load_gemini_api_key()

In [3]:
import json
import re
from dataclasses import dataclass
from enum import Enum
from typing import Any, Callable, Literal, TypedDict
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel, ConfigDict, Field, ValidationError


MODEL_NAME = "gemini-3.8-flash"

In [4]:
from google import genai


client = genai.Client(api_key=GEMINI_API_KEY)

## 3. Validate the user request early

Early input checks prevent obviously invalid requests from consuming model and tool resources. They should remain narrow and explainable; complex semantic decisions belong in a dedicated policy layer.

In [5]:
class UserRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    message: str = Field(min_length=1, max_length=2_000)
    user_id: str = Field(pattern=r"^[a-zA-Z0-9_-]{3,40}$")


def validate_user_request(data: dict[str, Any]) -> dict[str, Any]:
    try:
        request = UserRequest.model_validate(data)
        return {"valid": True, "request": request}
    except ValidationError as error:
        return {"valid": False, "errors": error.errors()}


print(validate_user_request({"message": "Check my account status.", "user_id": "user_17"}))
print(validate_user_request({"message": "", "user_id": "?"}))

{'valid': True, 'request': UserRequest(message='Check my account status.', user_id='user_17')}
{'valid': False, 'errors': [{'type': 'string_too_short', 'loc': ('message',), 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}, 'url': 'https://errors.pydantic.dev/2.13/v/string_too_short'}, {'type': 'string_pattern_mismatch', 'loc': ('user_id',), 'msg': "String should match pattern '^[a-zA-Z0-9_-]{3,40}$'", 'input': '?', 'ctx': {'pattern': '^[a-zA-Z0-9_-]{3,40}$'}, 'url': 'https://errors.pydantic.dev/2.13/v/string_pattern_mismatch'}]}


## 4. Attach policy metadata to tools

Risk is an application decision, not something the model should infer at execution time. We mark each tool with a risk level, side-effect status, and approval requirement.

In [6]:
class RiskLevel(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


@dataclass(frozen=True)
class Tool:
    name: str
    description: str
    arguments_model: type[BaseModel]
    function: Callable[..., Any]
    risk_level: RiskLevel
    has_side_effects: bool
    requires_approval: bool

    def schema(self) -> dict[str, Any]:
        return {
            "type": "function",
            "name": self.name,
            "description": self.description,
            "parameters": self.arguments_model.model_json_schema(),
        }

    def execute(self, arguments: dict[str, Any]) -> Any:
        validated = self.arguments_model.model_validate(arguments)
        return self.function(**validated.model_dump())

## 5. Define read-only and side-effecting tools

The notification function is simulated, but its policy matches a real external action. It also accepts an idempotency key. A production service should use this key to prevent the same action from being applied twice after retries or resumed execution.

In [7]:
class AccountStatusArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    account_id: str = Field(pattern=r"^ACC-[0-9]{4}$")


class NotificationArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    recipient: str = Field(pattern=r"^[a-zA-Z0-9_.-]{3,80}$")
    message: str = Field(min_length=1, max_length=500)
    idempotency_key: str = Field(min_length=8, max_length=80)


ACCOUNT_DATA = {
    "ACC-1001": "active",
    "ACC-1002": "review_required",
}
SENT_NOTIFICATIONS: dict[str, dict[str, str]] = {}


def get_account_status(account_id: str) -> dict[str, str]:
    return {
        "account_id": account_id,
        "status": ACCOUNT_DATA.get(account_id, "not_found"),
    }


def send_notification(
    recipient: str,
    message: str,
    idempotency_key: str,
) -> dict[str, str]:
    if idempotency_key in SENT_NOTIFICATIONS:
        return {
            **SENT_NOTIFICATIONS[idempotency_key],
            "delivery": "already_processed",
        }

    result = {
        "recipient": recipient,
        "message": message,
        "delivery": "simulated_success",
    }
    SENT_NOTIFICATIONS[idempotency_key] = result
    return result

In [8]:
tools = {
    "get_account_status": Tool(
        name="get_account_status",
        description="Read the current status of one account by account ID.",
        arguments_model=AccountStatusArguments,
        function=get_account_status,
        risk_level=RiskLevel.LOW,
        has_side_effects=False,
        requires_approval=False,
    ),
    "send_notification": Tool(
        name="send_notification",
        description=(
            "Send an external notification. This action requires explicit "
            "human approval before execution."
        ),
        arguments_model=NotificationArguments,
        function=send_notification,
        risk_level=RiskLevel.HIGH,
        has_side_effects=True,
        requires_approval=True,
    ),
}

tool_schemas = [tool.schema() for tool in tools.values()]
print([tool["name"] for tool in tool_schemas])

['get_account_status', 'send_notification']


## 6. Make policy decisions before execution

The policy function checks registration and approval requirements. It does not execute the function, which keeps authorization separate from the action itself.

In [9]:
def evaluate_tool_policy(tool_name: str) -> dict[str, Any]:
    tool = tools.get(tool_name)

    if tool is None:
        return {
            "decision": "deny",
            "reason": "Tool is not registered",
        }

    if tool.requires_approval:
        return {
            "decision": "require_approval",
            "reason": "Tool can create an external side effect",
            "risk_level": tool.risk_level.value,
        }

    return {
        "decision": "allow",
        "reason": "Registered low-risk read-only tool",
        "risk_level": tool.risk_level.value,
    }


print(evaluate_tool_policy("get_account_status"))
print(evaluate_tool_policy("send_notification"))
print(evaluate_tool_policy("delete_database"))

{'decision': 'allow', 'reason': 'Registered low-risk read-only tool', 'risk_level': 'low'}
{'decision': 'require_approval', 'reason': 'Tool can create an external side effect', 'risk_level': 'high'}
{'decision': 'deny', 'reason': 'Tool is not registered'}


## 7. Classify failures before retrying

Retrying every exception is unsafe. A temporary service outage may justify a bounded retry, while invalid arguments, permission denial, and approval rejection are permanent for the current request.

In [10]:
class TransientToolError(RuntimeError):
    pass


class PermanentToolError(RuntimeError):
    pass


def execute_with_retry(
    operation: Callable[[], Any],
    max_attempts: int = 3,
) -> dict[str, Any]:
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")

    for attempt in range(1, max_attempts + 1):
        try:
            return {
                "success": True,
                "result": operation(),
                "attempts": attempt,
            }
        except PermanentToolError as error:
            return {
                "success": False,
                "error_type": "permanent",
                "error": str(error),
                "attempts": attempt,
            }
        except TransientToolError as error:
            if attempt == max_attempts:
                return {
                    "success": False,
                    "error_type": "transient_exhausted",
                    "error": str(error),
                    "attempts": attempt,
                }

    raise RuntimeError("Unreachable retry state")

In [11]:
transient_attempts = {"count": 0}


def temporarily_unavailable_operation() -> str:
    transient_attempts["count"] += 1
    if transient_attempts["count"] < 3:
        raise TransientToolError("Temporary upstream failure")
    return "Recovered successfully"


print(execute_with_retry(temporarily_unavailable_operation, max_attempts=3))


def permanently_invalid_operation() -> str:
    raise PermanentToolError("Permission denied")


print(execute_with_retry(permanently_invalid_operation, max_attempts=3))

{'success': True, 'result': 'Recovered successfully', 'attempts': 3}
{'success': False, 'error_type': 'permanent', 'error': 'Permission denied', 'attempts': 1}


Real retry logic should also use exponential backoff and jitter for transient network failures. We omit waiting here so the notebook remains quick. Side-effecting operations should not be retried unless the external service provides an idempotent contract.

## 8. Build an interruptible approval graph

The graph accepts a proposed tool call, evaluates policy, pauses when approval is required, and then executes or rejects the action. `interrupt()` exposes a JSON-serializable review payload to the caller. Resuming with `Command(resume=True)` approves the action; `False` rejects it.

In [12]:
class ApprovalState(TypedDict):
    tool_name: str
    arguments: dict[str, Any]
    policy: dict[str, Any] | None
    approved: bool | None
    status: str
    observation: dict[str, Any] | None
    trace: list[dict[str, Any]]


def policy_node(state: ApprovalState) -> dict[str, Any]:
    policy = evaluate_tool_policy(state["tool_name"])
    return {
        "policy": policy,
        "trace": state["trace"] + [
            {"event": "policy_decision", "policy": policy}
        ],
    }


def route_policy(
    state: ApprovalState,
) -> Literal["approval", "execute", "reject"]:
    decision = state["policy"]["decision"]
    if decision == "require_approval":
        return "approval"
    if decision == "allow":
        return "execute"
    return "reject"

In [13]:
def approval_node(state: ApprovalState) -> dict[str, Any]:
    approved = interrupt(
        {
            "question": "Approve this tool execution?",
            "tool_name": state["tool_name"],
            "arguments": state["arguments"],
            "risk_level": state["policy"].get("risk_level"),
            "reason": state["policy"]["reason"],
        }
    )

    return {
        "approved": bool(approved),
        "trace": state["trace"] + [
            {"event": "human_decision", "approved": bool(approved)}
        ],
    }


def route_approval(
    state: ApprovalState,
) -> Literal["execute", "reject"]:
    return "execute" if state["approved"] else "reject"


def execute_node(state: ApprovalState) -> dict[str, Any]:
    tool = tools[state["tool_name"]]
    try:
        result = tool.execute(state["arguments"])
        observation = {"success": True, "result": result}
        status = "executed"
    except ValidationError as error:
        observation = {
            "success": False,
            "error_type": "invalid_arguments",
            "error": error.errors(),
        }
        status = "failed"
    except Exception as error:
        observation = {
            "success": False,
            "error_type": "execution_error",
            "error": f"{type(error).__name__}: {error}",
        }
        status = "failed"

    return {
        "status": status,
        "observation": observation,
        "trace": state["trace"] + [
            {"event": "tool_execution", "observation": observation}
        ],
    }


def reject_node(state: ApprovalState) -> dict[str, Any]:
    observation = {
        "success": False,
        "error_type": "policy_rejection",
        "error": "Tool execution was denied or not approved",
    }
    return {
        "status": "rejected",
        "observation": observation,
        "trace": state["trace"] + [
            {"event": "tool_rejected", "observation": observation}
        ],
    }

In [14]:
approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("policy", policy_node)
approval_builder.add_node("approval", approval_node)
approval_builder.add_node("execute", execute_node)
approval_builder.add_node("reject", reject_node)

approval_builder.add_edge(START, "policy")
approval_builder.add_conditional_edges(
    "policy",
    route_policy,
    {
        "approval": "approval",
        "execute": "execute",
        "reject": "reject",
    },
)
approval_builder.add_conditional_edges(
    "approval",
    route_approval,
    {
        "execute": "execute",
        "reject": "reject",
    },
)
approval_builder.add_edge("execute", END)
approval_builder.add_edge("reject", END)

approval_graph = approval_builder.compile(
    checkpointer=InMemorySaver()
)

## 9. Execute a low-risk request automatically

The read-only status tool passes policy and reaches the execution node without interruption.

In [15]:
low_risk_state: ApprovalState = {
    "tool_name": "get_account_status",
    "arguments": {"account_id": "ACC-1001"},
    "policy": None,
    "approved": None,
    "status": "pending",
    "observation": None,
    "trace": [],
}

low_risk_config = {
    "configurable": {"thread_id": "low-risk-example"}
}

low_risk_result = approval_graph.invoke(
    low_risk_state,
    config=low_risk_config,
)

print(low_risk_result)

{'tool_name': 'get_account_status', 'arguments': {'account_id': 'ACC-1001'}, 'policy': {'decision': 'allow', 'reason': 'Registered low-risk read-only tool', 'risk_level': 'low'}, 'approved': None, 'status': 'executed', 'observation': {'success': True, 'result': {'account_id': 'ACC-1001', 'status': 'active'}}, 'trace': [{'event': 'policy_decision', 'policy': {'decision': 'allow', 'reason': 'Registered low-risk read-only tool', 'risk_level': 'low'}}, {'event': 'tool_execution', 'observation': {'success': True, 'result': {'account_id': 'ACC-1001', 'status': 'active'}}}]}


## 10. Pause a side effect for review

The first invocation stops at `interrupt()`. With the default `invoke` interface, pending interrupts appear under `__interrupt__`.

In [16]:
notification_state: ApprovalState = {
    "tool_name": "send_notification",
    "arguments": {
        "recipient": "course-admin",
        "message": "Notebook validation completed successfully.",
        "idempotency_key": f"tutorial-{uuid4().hex}",
    },
    "policy": None,
    "approved": None,
    "status": "pending",
    "observation": None,
    "trace": [],
}

approval_config = {
    "configurable": {"thread_id": "approval-example-001"}
}

paused_result = approval_graph.invoke(
    notification_state,
    config=approval_config,
)

print(paused_result["__interrupt__"])

[Interrupt(value={'question': 'Approve this tool execution?', 'tool_name': 'send_notification', 'arguments': {'recipient': 'course-admin', 'message': 'Notebook validation completed successfully.', 'idempotency_key': 'tutorial-760820a2933b402691d986715c6c75e7'}, 'risk_level': 'high', 'reason': 'Tool can create an external side effect'}, id='051347d2911c14de327266d3b07fa6fe', response_schema=None)]


Review the tool name, arguments, risk level, and reason before resuming. The human decision originates outside the model and is associated with the same thread ID.

In [17]:
approved_result = approval_graph.invoke(
    Command(resume=True),
    config=approval_config,
)

print(approved_result["status"])
print(approved_result["observation"])
print(approved_result["trace"])

executed
{'success': True, 'result': {'recipient': 'course-admin', 'message': 'Notebook validation completed successfully.', 'delivery': 'simulated_success'}}
[{'event': 'policy_decision', 'policy': {'decision': 'require_approval', 'reason': 'Tool can create an external side effect', 'risk_level': 'high'}}, {'event': 'human_decision', 'approved': True}, {'event': 'tool_execution', 'observation': {'success': True, 'result': {'recipient': 'course-admin', 'message': 'Notebook validation completed successfully.', 'delivery': 'simulated_success'}}}]


To test rejection, start a new thread with the same proposed action and resume it with `Command(resume=False)`. Reusing a completed thread would continue its existing checkpoint lineage rather than creating a fresh approval request.

## 11. Inspect output before returning it

Output guardrails are a final defensive layer. This simple redactor handles API-key-like patterns for demonstration; production secret detection should use a maintained scanner and should also prevent secrets from entering model context in the first place.

In [18]:
SECRET_PATTERNS = [
    re.compile(r"AIza[0-9A-Za-z_-]{20,}"),
    re.compile(r"sk-[0-9A-Za-z_-]{20,}"),
]


def redact_secrets(text: str) -> dict[str, Any]:
    redacted = text
    match_count = 0

    for pattern in SECRET_PATTERNS:
        redacted, count = pattern.subn("[REDACTED_SECRET]", redacted)
        match_count += count

    return {
        "text": redacted,
        "redactions": match_count,
        "safe_to_return": match_count == 0,
    }


example_output = "Debug key: sk-abcdefghijklmnopqrstuvwxyz123456"
print(redact_secrets(example_output))

{'text': 'Debug key: [REDACTED_SECRET]', 'redactions': 1, 'safe_to_return': False}


## 12. Integrate a real model proposal

Gemini receives both tool schemas. It proposes a function call, but the action still passes through our policy graph. The model cannot approve its own side-effecting request.

In [19]:
user_request = (
    "Send a notification to course-admin saying that the live agent test "
    "completed. Use idempotency key tutorial-live-001."
)

proposal = client.interactions.create(
    model=MODEL_NAME,
    input=user_request,
    tools=tool_schemas,
)

proposed_calls = [
    step for step in proposal.steps
    if step.type == "function_call"
]

if not proposed_calls:
    raise RuntimeError("Gemini did not propose a tool call")

proposed_call = proposed_calls[0]
print(proposed_call.name, proposed_call.arguments)

send_notification {'recipient': 'course-admin', 'idempotency_key': 'tutorial-live-001', 'message': 'The live agent test completed.'}


In [20]:
model_approval_state: ApprovalState = {
    "tool_name": proposed_call.name,
    "arguments": dict(proposed_call.arguments),
    "policy": None,
    "approved": None,
    "status": "pending",
    "observation": None,
    "trace": [],
}

model_approval_config = {
    "configurable": {"thread_id": f"model-approval-{uuid4().hex}"}
}

model_paused = approval_graph.invoke(
    model_approval_state,
    config=model_approval_config,
)

print(model_paused["__interrupt__"])

[Interrupt(value={'question': 'Approve this tool execution?', 'tool_name': 'send_notification', 'arguments': {'recipient': 'course-admin', 'idempotency_key': 'tutorial-live-001', 'message': 'The live agent test completed.'}, 'risk_level': 'high', 'reason': 'Tool can create an external side effect'}, id='3742f1d30e9823e9a316e66a99536f1f', response_schema=None)]


In an interactive application, the interrupt payload would be rendered in an approval UI. For the tutorial, inspect it manually before running the next cell.

In [21]:
model_approved = approval_graph.invoke(
    Command(resume=True),
    config=model_approval_config,
)

final_interaction = client.interactions.create(
    model=MODEL_NAME,
    previous_interaction_id=proposal.id,
    tools=tool_schemas,
    input=[
        {
            "type": "function_result",
            "name": proposed_call.name,
            "call_id": proposed_call.id,
            "result": [
                {
                    "type": "text",
                    "text": json.dumps(model_approved["observation"]),
                }
            ],
        }
    ],
)

checked_output = redact_secrets(final_interaction.output_text or "")
print(checked_output["text"])

The notification has been successfully sent to `course-admin` with the message: *"The live agent test completed."* (Idempotency key: `tutorial-live-001`).


## 13. Important interrupt rules

LangGraph restarts an interrupted node from its beginning when execution resumes. Code before `interrupt()` may therefore run again. Keep side effects after approval, do not wrap `interrupt()` in a broad `try/except`, and pass only JSON-serializable values in the interrupt payload.

The approval node in this notebook performs no external action before the interrupt. Execution occurs in a separate node after approval, which makes replay behavior easier to reason about.

## 14. Exercise: test the rejection path

Start a new approval thread for `send_notification`, confirm that it pauses, and resume with `Command(resume=False)`. Verify that:

- the final status is `rejected`;
- the observation has `error_type="policy_rejection"`;
- no new entry was added to `SENT_NOTIFICATIONS`; and
- the trace contains a human rejection followed by a tool-rejected event.

In [ ]:
# Create a new notification state and unique thread ID.
# Invoke until interruption, then resume with Command(resume=False).

## Notebook summary

We implemented layered guardrails rather than relying on one safety prompt. Requests and arguments are schema-validated, tool risk is declared by application policy, permanent failures are not retried, transient retries are bounded, side effects require approval, idempotency reduces duplicate actions, and final output is inspected before release.

The next notebook will evaluate the system systematically. We will build fixed test cases for answer quality, tool choice, arguments, retrieval, citations, latency, step counts, failures, and cost-related metadata.

### References

- [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling)